# Squirrel detection · RF-DETR Medium

One notebook: **secure setup → COCO download → audit & EDA → ClearML → GPU probe → 50-epoch training → validation → test → exported weights**.

**Decisions:** one class (`SQUIRREL`); seed 42; leak-free repaired 70:15:15 splits; leakage blocks training; batch/accumulation sequence **8×2, 4×4, 4×2**; measured baseline rather than promised accuracy. Version 6: [Root and Nut dataset](https://universe.roboflow.com/root-and-nut/squirrel-re-id-training-v1-fzpbr/dataset/6), CC BY 4.0. Counts are measured after download. This is detection, not species classification or individual re-identification.

## Run instructions
From the repository: `uv sync --locked`, then `uv run jupyter lab squirrel_detection_rfdetr_medium.ipynb`. Select its Python environment and run cells in order. Credentials belong only in `.env`; never paste them into cells. The install is reproducible through `uv.lock`. The notebook does not install packages into a running kernel.

Reports and checkpoints are generated locally under `output/`; dataset under `data/`. All workflow code is below. A stopped audit must be resolved before training. Resume by setting `RESUME_RUN` to an existing run directory; it loads `last.ckpt` and the saved batch setting, keeping the original 50-epoch ceiling.

### Milestone criteria
1. Setup: pinned imports, credentials configured, CUDA forward/backward works.
2. Data: complete download, all images/boxes valid, no confirmed or unresolved cross-split leakage.
3. EDA: counts reconcile, distributions and annotation galleries rendered.
4. Tracking/probe: ClearML receives a scalar and full training/validation steps fit memory.
5. Training: finite losses, epoch metrics, checkpoints and progress galleries recorded.
6. Evaluation: best validation checkpoint, validation-selected threshold frozen for test; honest undefined metrics.
7. Export: copied weights reload and reproduce predictions, with metadata and checksum.

### Clean 70:15:15 Dataset Repair
The dataset splits have been repaired into a balanced, leak-free **70 : 15 : 15** distribution (**6,495 train / 1,392 validation / 1,393 test images** across 9,280 unique images, deduplicating 292 exact identical copies).
All Roboflow augmentations (horizontal flips, 90° rotations, brightness/exposure) sharing the same filename prefix and all temporal burst camera sequences (pHash Hamming distance ≤ 4) have been grouped into connected components. To prevent evaluation skew, all large bursts (> 4 images) are confined to `train`, while `valid` and `test` each contain over 510 completely independent scenes/clusters. Cross-split candidate leakage pairs: **0**.

In [ ]:
# 1. Configuration — edit only nonsecret experiment settings here.
from pathlib import Path
import os, sys, json, re, io, time, uuid, hashlib, shutil, zipfile, contextlib
import math, random, gc, logging, traceback, html, platform, importlib.metadata
from datetime import datetime, timezone
ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / 'pyproject.toml').exists()), None)
assert ROOT is not None, 'Launch Jupyter from the repository.'
CFG = dict(seed=42, workspace='root-and-nut', project='squirrel-re-id-training-v1-fzpbr',
           version=6, epochs=50, resolution=576, lr=1e-4, eval_batch_size=2,
           batch_candidates=[[8, 2], [4, 4], [4, 2]], clearml_project='CSCI-635-Squirrel-Detection',
           prediction_floor=0.0001, iou=0.5, max_detections=100,
           leakage_max_distance=4,  # pHash Hamming threshold knob (0=exact pHash, 2=tight burst, 4=balanced)
           use_repaired_splits=True)  # True = use leak-free repaired splits (data/squirrel-v6-clean)
RESUME_RUN = None  # e.g. ROOT / 'output/runs/<run-id>'; never resume from lightweight .pth.
# Review-only exclusions: pair_id -> written reason why two candidates are NOT related.
# Confirmed duplicates/source siblings cannot be waived here.
LEAKAGE_FALSE_POSITIVES = {}
DATA_RAW = ROOT / 'data/squirrel-v6-coco'
DATA_CLEAN = ROOT / 'data/squirrel-v6-clean'
DATA = DATA_CLEAN if (CFG.get('use_repaired_splits', True) and (DATA_CLEAN / '.complete.json').exists()) else DATA_RAW
RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ') + '-' + uuid.uuid4().hex[:6]
RUN = Path(RESUME_RUN).resolve() if RESUME_RUN else ROOT / 'output/runs' / RUN_ID
REPORT = RUN / 'reports'
for path in (DATA.parent, REPORT): path.mkdir(parents=True, exist_ok=True)
os.environ['RF_HOME'] = str(ROOT/'output/pretrained')
from dotenv import load_dotenv
load_dotenv(ROOT / '.env', override=False)
SECRET_NAMES = ('ROBOFLOW_KEY', 'ROBOFLOW_API_KEY', 'CLEARML_API_ACCESS_KEY', 'CLEARML_API_SECRET_KEY', 'CLEARML_TOKEN')
from urllib.parse import quote, quote_plus
_SECRETS = sorted({v for k in SECRET_NAMES if os.environ.get(k) for v in (os.environ[k], quote(os.environ[k],safe=''), quote_plus(os.environ[k]))}, key=len, reverse=True)
def redact(value):
    text = str(value)
    for secret in sorted(_SECRETS, key=len, reverse=True):
        text = text.replace(secret, '[REDACTED]')
    text = re.sub(r'https?://[^\s\"\'<>]+', lambda m: m[0].split('?')[0] + ('?[REDACTED]' if '?' in m[0] else ''), text)
    return text

def assert_secret_free(value):
    text = value if isinstance(value, str) else json.dumps(value, default=str)
    if any(s in text for s in _SECRETS):
        raise RuntimeError('Secret scan failed; payload was not saved or uploaded.') from None

def save_json(path, value):
    text = json.dumps(value, indent=2, default=str, allow_nan=False)
    assert_secret_free(text)
    Path(path).parent.mkdir(parents=True, exist_ok=True)
    Path(path).write_text(text)

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(1024 * 1024), b''): h.update(block)
    return h.hexdigest()

class SafeWriter(io.TextIOBase):
    # Buffer complete lines so secrets split across SDK writes are also redacted.
    def __init__(self, target): self.target, self.pending = target, ''
    def write(self, text):
        self.pending += text
        while '\n' in self.pending:
            line, self.pending = self.pending.split('\n', 1)
            safe = redact(line) + '\n'
            self.target.write(safe)
            with (RUN / 'console.log').open('a') as f: f.write(safe)
        return len(text)
    def flush(self): self.target.flush()
    def finish(self):
        if self.pending:
            self.write('\n')
        self.flush()

@contextlib.contextmanager
def safe_stage(name):
    out, err = SafeWriter(sys.stdout), SafeWriter(sys.stderr)
    try:
        with contextlib.redirect_stdout(out), contextlib.redirect_stderr(err): yield
    except Exception as exc:
        # No raw SDK exception traceback: signed request URLs can contain secrets.
        summary = redact(f'{name}: {type(exc).__name__}: {exc}')
        with (RUN / 'errors.log').open('a') as f: f.write(summary + '\n')
        raise RuntimeError(summary) from None
    finally:
        out.finish(); err.finish()

required = ['CLEARML_API_ACCESS_KEY', 'CLEARML_API_SECRET_KEY']
missing = [k for k in required if not os.getenv(k)]
if not (os.getenv('ROBOFLOW_KEY') or os.getenv('ROBOFLOW_API_KEY')): missing.append('ROBOFLOW_KEY')
assert not missing, 'Missing credential variable names: ' + ', '.join(missing)
# Disable automatic environment/repository/notebook collection before importing ClearML.
os.environ.pop('__SQUIRREL_NO_ENV_CAPTURE__', None)
os.environ['CLEARML_LOG_ENVIRONMENT'] = '__SQUIRREL_NO_ENV_CAPTURE__'
os.environ['CLEARML_NO_DEFAULT_SERVER'] = '1'
os.environ['CLEARML_API_HOST'] = os.getenv('CLEARML_API_HOST', 'https://api.clear.ml')
os.environ['CLEARML_WEB_HOST'] = os.getenv('CLEARML_WEB_HOST', 'https://app.clear.ml')
os.environ['CLEARML_FILES_HOST'] = os.getenv('CLEARML_FILES_HOST', 'https://files.clear.ml')
os.environ['CLEARML_AGENT_LOG_ENVIRONMENT'] = ''
print('Credential names validated; values hidden. Run:', RUN.name)

In [ ]:
# Imports and actual GPU test (no model/dataset download here).
with safe_stage('environment'):
    import numpy as np, pandas as pd, matplotlib.pyplot as plt
    import torch, requests, imagehash
    from PIL import Image, ImageDraw
    from IPython.display import display, HTML
    from sklearn.metrics import roc_curve, roc_auc_score, confusion_matrix
    from pycocotools.coco import COCO
    from pycocotools.cocoeval import COCOeval
    from rfdetr import RFDETRMedium
    from rfdetr.config import RFDETRMediumConfig, TrainConfig
    from rfdetr.training import RFDETRDataModule, RFDETRModelModule, build_trainer
    from pytorch_lightning import Callback, seed_everything
    assert importlib.metadata.version('rfdetr') == '1.10.1', 'Run uv sync --locked.'
    assert torch.cuda.is_available(), 'A CUDA GPU is required for this workflow.'
    seed_everything(CFG['seed'], workers=True)
    x = torch.randn(64, 64, device='cuda', requires_grad=True)
    x.square().mean().backward(); torch.cuda.synchronize(); del x
    versions = {p: importlib.metadata.version(p) for p in ['rfdetr','torch','torchvision','clearml','roboflow','numpy','pycocotools']}
    environment = dict(python=platform.python_version(), packages=versions, cuda=torch.version.cuda,
                       gpu=torch.cuda.get_device_name(), free_vram_gib=torch.cuda.mem_get_info()[0] / 2**30)
    save_json(RUN / 'environment.json', environment)
    if RESUME_RUN:
        saved = json.loads((RUN / 'config.json').read_text())
        assert saved == CFG, 'Resume requires identical configuration.'
    else: save_json(RUN / 'config.json', CFG)
    print(environment)

## 2. Secure, restartable download and integrity audit
The export endpoint matches the Roboflow SDK, but download streaming is implemented here to avoid SDK error messages containing signed URLs, check available disk space, validate ZIP paths, and mark completion only after successful extraction. Raw source files are preserved. No archive-size cap applies; 5 GiB is reserved for operational headroom, followed by a separate checkpoint-space check.

In [ ]:
def check_space(required=0, where=DATA.parent):
    free = shutil.disk_usage(where).free
    if free < required + 5 * 2**30:
        raise RuntimeError('Insufficient disk space; free additional storage before rerunning.')

def download_dataset():
    marker = DATA / '.complete.json'
    if marker.exists():
        manifest = json.loads(marker.read_text())
        assert manifest['version'] == CFG['version']
        for split, digest in manifest['annotation_hashes'].items():
            assert sha256(DATA / split / '_annotations.coco.json') == digest, 'Cached annotations changed.'
        print('Reusing verified completed export. Images will be audited again.')
        return manifest
    if DATA.exists(): raise RuntimeError('Incomplete dataset directory exists; inspect it before removing/retrying.')
    staging = DATA.parent / (DATA.name + '.partial')
    staging.mkdir(exist_ok=True)
    archive = staging / 'dataset.zip'
    key = os.getenv('ROBOFLOW_KEY') or os.environ['ROBOFLOW_API_KEY']
    endpoint = f"https://api.roboflow.com/{CFG['workspace']}/{CFG['project']}/{CFG['version']}/coco"
    link = None
    for attempt in range(60):
        response = requests.get(endpoint, params={'api_key': key}, timeout=(15, 90))
        if response.status_code not in (200, 202):
            raise RuntimeError(f'Roboflow export HTTP {response.status_code}; response suppressed.')
        payload = response.json()
        if payload.get('ready') is not False and payload.get('export', {}).get('link'):
            link = payload['export']['link']; break
        time.sleep(5)
    if not link: raise RuntimeError('Export is not ready; rerun this cell later.')
    if not link.startswith('https://'): raise RuntimeError('Refusing non-HTTPS export link.')
    check_space()
    with requests.get(link, stream=True, timeout=(15, 120)) as response:
        if response.status_code != 200: raise RuntimeError(f'Download HTTP {response.status_code}; URL suppressed.')
        length = int(response.headers.get('Content-Length', 0))
        check_space(length)
        downloaded = 0
        with archive.open('wb') as f:
            for chunk in response.iter_content(1024 * 1024):
                if not chunk: continue
                check_space(len(chunk)); f.write(chunk); downloaded += len(chunk)
        if length and downloaded != length: raise RuntimeError('Incomplete archive; rerun download.')
    archive_hash = sha256(archive)
    extracted = staging / 'extracted'
    if extracted.exists(): shutil.rmtree(extracted)
    extracted.mkdir()
    with zipfile.ZipFile(archive) as z:
        check_space(sum(i.file_size for i in z.infolist()))
        for info in z.infolist():
            target = (extracted / info.filename).resolve()
            if not target.is_relative_to(extracted.resolve()): raise RuntimeError('Unsafe ZIP path.')
            if ((info.external_attr >> 16) & 0o170000) == 0o120000: raise RuntimeError('ZIP symlink rejected.')
            check_space(info.file_size)
            z.extract(info, extracted)
    assert all((extracted / s / '_annotations.coco.json').exists() for s in ['train','valid','test']), 'Missing COCO splits.'
    manifest = dict(version=6, workspace=CFG['workspace'], project=CFG['project'], format='coco',
                    license='CC BY 4.0', source='https://universe.roboflow.com/root-and-nut/squirrel-re-id-training-v1-fzpbr/dataset/6',
                    archive_sha256=archive_hash, archive_bytes=downloaded,
                    annotation_hashes={s:sha256(extracted / s / '_annotations.coco.json') for s in ['train','valid','test']})
    extracted.rename(DATA)
    save_json(marker, manifest)
    archive.unlink(); staging.rmdir()
    return manifest

with safe_stage('download'):
    manifest = download_dataset()
    save_json(RUN / 'dataset_manifest.json', manifest)
    print('Dataset downloaded:', DATA)

In [ ]:
def xywh_to_xyxy(box):
    x, y, w, h = map(float, box)
    return [x, y, x+w, y+h]

def audit_dataset():
    splits, records, boxes, problems = {}, [], [], []
    for split in ['train','valid','test']:
        doc = json.loads((DATA / split / '_annotations.coco.json').read_text())
        assert doc['images'], f'{split}: empty split'
        assert len({i['id'] for i in doc['images']}) == len(doc['images']), 'Duplicate image IDs'
        assert len({a['id'] for a in doc['annotations']}) == len(doc['annotations']), 'Duplicate annotation IDs'
        categories = {c['id']: c['name'] for c in doc['categories']}
        used = {a['category_id'] for a in doc['annotations']}
        assert used and all(categories[k].upper() == 'SQUIRREL' for k in used), 'Expected SQUIRREL only'
        assert len(used) == 1, 'Multiple foreground category IDs'
        split_images = {}
        for item in doc['images']:
            path = (DATA / split / item['file_name']).resolve()
            assert path.is_relative_to((DATA / split).resolve()), 'Unsafe image path'
            try:
                with Image.open(path) as im:
                    im.load(); rgb = im.convert('RGB'); width, height = rgb.size
                    assert (width, height) == (item['width'], item['height'])
                    pixel_hash = hashlib.sha256(f'{width}x{height}'.encode() + rgb.tobytes()).hexdigest()
                    phash = str(imagehash.phash(rgb))
                # Roboflow augmentation suffix; do not infer unrelated camera filenames as groups.
                source = re.sub(r'\.rf\..*$', '', item['file_name'])
                row = dict(split=split, image_id=item['id'], file_name=item['file_name'], path=str(path),
                           width=width, height=height, pixel_hash=pixel_hash, phash=phash,
                           source=source, objects=0, gt=[])
                split_images[item['id']] = row
            except Exception:
                problems.append(f'{split}: image {item["id"]} unreadable or dimensions inconsistent')
        for ann in doc['annotations']:
            row = split_images.get(ann['image_id'])
            if row is None: problems.append(f'{split}: orphan annotation {ann["id"]}'); continue
            b = ann.get('bbox', [])
            valid = len(b)==4 and all(math.isfinite(float(x)) for x in b)
            if valid:
                x,y,w,h = map(float,b)
                valid = x >= 0 and y >= 0 and w > 0 and h > 0 and x+w <= row['width']+0.01 and y+h <= row['height']+0.01
            if not valid: problems.append(f'{split}: invalid box {ann["id"]}'); continue
            if ann.get('iscrowd',0) or ann.get('ignore',0):
                problems.append(f'{split}: crowd/ignored annotation needs explicit metric policy: {ann["id"]}'); continue
            row['objects'] += 1; row['gt'].append(xywh_to_xyxy(b))
            boxes.append(dict(split=split, image_id=row['image_id'], area=w*h, width=w, height=h,
                              aspect=w/h, cx=(x+w/2)/row['width'], cy=(y+h/2)/row['height'],
                              relative_area=w*h/(row['width']*row['height'])))
        records.extend(split_images.values()); splits[split] = doc
    save_json(REPORT / 'integrity.json', {'problems':problems})
    if problems: raise RuntimeError(f'{len(problems)} data-integrity issues; see integrity.json. No training permitted.')
    return splits, records, pd.DataFrame(boxes)

def find_leakage_pairs(records, max_distance=None):
    if max_distance is None:
        max_distance = CFG.get('leakage_max_distance', 4) if 'CFG' in globals() else 4
    # Compare each cross-split pair; ~10k images fits a compact vectorized hash comparison.
    hashes = np.array([int(r['phash'],16) for r in records], dtype=np.uint64)
    split_arr = np.array([r['split'] for r in records])
    popcount = np.array([int(x).bit_count() for x in range(256)], dtype=np.uint8)
    pairs = []
    for i, r in enumerate(records):
        idx = np.flatnonzero((np.arange(len(records)) > i) & (split_arr != r['split']))
        if not len(idx): continue
        xor = (hashes[idx] ^ hashes[i]).view(np.uint8).reshape(-1,8)
        distances = popcount[xor].sum(axis=1)
        # Near-identical pHash <= max_distance candidates; exact pixel duplicates also checked.
        near = set(idx[distances <= max_distance].tolist())
        for j in idx:
            if records[j]['pixel_hash'] == r['pixel_hash']: near.add(int(j))
        for j in sorted(near):
            other = records[j]
            dist = (int(hashes[i]) ^ int(hashes[j])).bit_count()
            exact_pixels = other['pixel_hash'] == r['pixel_hash']
            same_source = other['source'] == r['source'] and dist <= max_distance
            confirmed = exact_pixels or same_source
            pair_id = hashlib.sha256((r['split']+'/'+r['file_name']+r['pixel_hash']+'|'+other['split']+'/'+other['file_name']+other['pixel_hash']).encode()).hexdigest()[:16]
            pairs.append(dict(pair_id=pair_id, left=i, right=j, confirmed=confirmed,
                              reason='identical pixels/source' if confirmed else 'perceptual similarity',
                              distance=dist))
    return pairs

with safe_stage('audit'):
    splits, records, box_df = audit_dataset()
    image_df = pd.DataFrame([{k:v for k,v in r.items() if k != 'gt'} for r in records])
    image_df.to_csv(REPORT / 'image_inventory.csv', index=False)
    box_df.to_csv(REPORT / 'box_inventory.csv', index=False)
    pairs = find_leakage_pairs(records)
    save_json(REPORT / 'leakage_pairs.json', pairs)
    leakage_fingerprint = hashlib.sha256(json.dumps({'manifest':manifest,'images':[(r['split'],r['file_name'],r['pixel_hash']) for r in records]},sort_keys=True).encode()).hexdigest()
    if RESUME_RUN and (REPORT/'leakage_decisions.json').exists():
        assert json.loads((REPORT/'leakage_decisions.json').read_text())['dataset_fingerprint']==leakage_fingerprint, 'Dataset changed since original run.'
    print('Integrity passed. Cross-split candidate pairs:',len(pairs))

## 3. EDA and leakage review
Green boxes show ground truth. Source/burst metadata may be absent: perceptual similarity is a screening tool, not proof of independence. The review file records dataset hashes and reasons for false-positive decisions. Confirmed duplicates require correcting the dataset/splits externally; this notebook never silently changes the supplied split.

In [ ]:
def save_figure(fig, name, directory=REPORT):
    directory.mkdir(parents=True, exist_ok=True)
    fig.savefig(directory / (name+'.png'), dpi=150, bbox_inches='tight')
    display(fig); plt.close(fig)

def draw_sample(row, predictions=None, threshold=0):
    im = Image.open(row['path']).convert('RGB'); draw=ImageDraw.Draw(im)
    for b in row['gt']: draw.rectangle(b, outline='lime', width=3)
    if predictions:
        for p in predictions:
            if p['score'] < threshold: continue
            draw.rectangle(p['box'], outline='red', width=3)
            draw.text((p['box'][0],max(0,p['box'][1]-12)),f"SQUIRREL {p['score']:.2f}", fill='red')
    im.thumbnail((640,480)); return im

def gallery(rows, name, predictions=None, threshold=0, directory=REPORT):
    if not rows: return
    fig, axes=plt.subplots(math.ceil(len(rows)/3),3,figsize=(15,4*math.ceil(len(rows)/3)),squeeze=False)
    for ax in axes.flat: ax.axis('off')
    for ax,row in zip(axes.flat,rows):
        ax.imshow(draw_sample(row, (predictions or {}).get(row['image_id']),threshold))
        ax.set_title(f"{row['split']} · image {row['image_id']} · {row['objects']} GT")
    save_figure(fig,name,directory)

def html_report(directory,title,summary):
    assert_secret_free(summary)
    images=''.join(f'<figure><figcaption>{html.escape(p.stem)}</figcaption><img src="{p.name}" style="max-width:100%"></figure>' for p in sorted([*directory.glob('*.png'),*directory.glob('*.jpg')]))
    links=''.join(f'<li><a href="{p.relative_to(directory).as_posix()}">{html.escape(p.parent.name)} report</a></li>' for p in sorted(directory.glob('*/report.html')))
    text=f'<!doctype html><meta charset="utf-8"><title>{html.escape(title)}</title><body style="max-width:1100px;margin:auto;font:16px system-ui"><h1>{html.escape(title)}</h1><ul>{links}</ul><pre style="white-space:pre-wrap">{html.escape(json.dumps(summary,indent=2,default=str))}</pre>{images}</body>'
    assert_secret_free(text); (directory/'report.html').write_text(text)

summary = image_df.groupby('split').agg(images=('image_id','count'),objects=('objects','sum'),negative_images=('objects',lambda x:int((x==0).sum())))
display(summary); summary.to_csv(REPORT/'split_summary.csv')
fig,axes=plt.subplots(2,3,figsize=(16,9))
summary[['images','objects','negative_images']].plot.bar(ax=axes[0,0],title='Split counts')
for split in ['train','valid','test']:
    d=image_df[image_df.split==split]; b=box_df[box_df.split==split]
    axes[0,1].scatter(d.width,d.height,s=5,alpha=.25,label=split)
    axes[0,2].hist(d.objects,bins=range(int(image_df.objects.max())+2),alpha=.4,label=split)
    axes[1,0].hist(np.log10(b.area),bins=30,alpha=.4,label=split)
    axes[1,1].hist(b.aspect,bins=30,alpha=.4,label=split)
axes[0,1].set(title='Image dimensions',xlabel='Width',ylabel='Height')
axes[0,2].set(title='Objects per image'); axes[1,0].set(title='Box area (log10 pixels²)')
axes[1,1].set(title='Box aspect ratio')
axes[1,2].hist2d(box_df.cx,box_df.cy,bins=30); axes[1,2].set(title='Normalized box centers',xlabel='x',ylabel='y')
for ax in axes.flat[:5]: ax.legend()
fig.tight_layout(); save_figure(fig,'eda_distributions')
box_df['size']=pd.cut(box_df.area,[0,32**2,96**2,np.inf],labels=['small','medium','large'],right=False)
size_counts=pd.crosstab(box_df.split,box_df['size']); display(size_counts)
size_counts.to_csv(REPORT/'object_sizes.csv')
fig,axes=plt.subplots(1,2,figsize=(12,4))
for split in ['train','valid','test']:
    d=image_df[image_df.split==split]
    axes[0].hist(d.width/d.height,bins=30,alpha=.4,label=split)
    axes[1].hist(box_df[box_df.split==split].relative_area,bins=30,alpha=.4,label=split)
axes[0].set_title('Image aspect ratios'); axes[1].set_title('Relative box area')
for ax in axes: ax.legend()
save_figure(fig,'eda_aspect_relative_area')
rng=random.Random(CFG['seed'])
for split in ['train','valid','test']:
    rows=[r for r in records if r['split']==split]
    gallery(rng.sample(rows,min(9,len(rows))),f'samples_{split}')
negative_rows=[r for r in records if r['objects']==0]
gallery(rng.sample(negative_rows,min(9,len(negative_rows))),'negative_samples')
# Every candidate is rendered in paginated contact sheets; IDs map to leakage_pairs.json.
review_dir=REPORT/'leakage_review'; review_dir.mkdir(exist_ok=True)
for start in range(0,len(pairs),12):
    page=pairs[start:start+12]
    fig,axes=plt.subplots(len(page),2,figsize=(10,3*len(page)),squeeze=False)
    for axis,pair in zip(axes,page):
        for ax,side in zip(axis,['left','right']):
            ax.imshow(draw_sample(records[pair[side]])); ax.axis('off')
            ax.set_title(f"{pair['pair_id']} · {records[pair[side]]['split']} · {pair['reason']}")
    fig.tight_layout(); fig.savefig(review_dir/f'pairs_{start:05d}.jpg',dpi=75,pil_kwargs={'quality':80}); plt.close(fig)
html_report(review_dir,'Cross-split leakage review',{'pairs':len(pairs),'fingerprint':leakage_fingerprint})
eda_summary={'splits':summary.to_dict(orient='index'),'leakage_candidates':len(pairs),
             'limitations':'Burst/camera groups unavailable unless encoded in filenames; pHash audit cannot establish independence.',
             'license':'CC BY 4.0; Root and Nut, Roboflow Universe version 6'}
html_report(REPORT,'Squirrel dataset EDA',eda_summary)
print('EDA report:',REPORT/'report.html')

## 4. ClearML and safe experiment logging
A fresh task is created for each execution (resume tasks link to the previous task). Automatic framework, notebook/repository, argument, environment, and console capture are disabled. Only explicit nonsecret parameters, numeric metrics, sanitized logs, reports, and the final model are uploaded. Local metrics remain authoritative if connectivity fails; pending events are retained for replay.

In [ ]:
from clearml import Task, OutputModel
with safe_stage('ClearML setup'):
    task=Task.init(project_name=CFG['clearml_project'],task_name=f'rfdetr-medium-v6-{RUN.name}',
                   reuse_last_task_id=False,output_uri=True,auto_connect_arg_parser=False,
                   auto_connect_frameworks={name:False for name in ['detect_repository','pytorch','tensorboard','tensorflow',
                         'matplotlib','scikit','joblib','hydra','tfdefines','megengine','xgboost','catboost','fastai','lightgbm','gradio']},
                   auto_connect_streams=False,auto_resource_monitoring=False)
    task.connect(dict(CFG),name='Experiment')
    if (RUN/'clearml.json').exists():
        task.connect({'previous_task_id':json.loads((RUN/'clearml.json').read_text())['task_id']},name='Resume')
    save_json(RUN/'clearml.json',{'task_id':task.id,'url':task.get_output_log_web_page()})
    logger=task.get_logger()
    logger.report_scalar('preflight','connected',value=1,iteration=0)
    task.flush(wait_for_uploads=True)
    print('ClearML task:',task.get_output_log_web_page())

def emit_scalar(title,series,value,step):
    if not math.isfinite(float(value)): return
    event=dict(title=title,series=series,value=float(value),iteration=int(step))
    assert_secret_free(event)
    with (RUN/'scalars.jsonl').open('a') as f: f.write(json.dumps(event)+'\n')
    try: logger.report_scalar(**event)
    except Exception:
        with (RUN/'pending_scalars.jsonl').open('a') as f: f.write(json.dumps(event)+'\n')

def upload_artifact(name,path):
    path=Path(path)
    if path.suffix in {'.json','.jsonl','.csv','.html','.log','.txt'}: assert_secret_free(path.read_text())
    try: task.upload_artifact(name=name,artifact_object=str(path),wait_on_upload=True)
    except Exception:
        with (RUN/'pending_artifacts.jsonl').open('a') as f: f.write(json.dumps({'name':name,'path':str(path)})+'\n')

In [ ]:
# Hard gate, deliberately separate from EDA so reports remain available on failure.
confirmed=[p for p in pairs if p['confirmed']]
unresolved=[p for p in pairs if not p['confirmed'] and not str(LEAKAGE_FALSE_POSITIVES.get(p['pair_id'],'')).strip()]
save_json(REPORT/'leakage_decisions.json',{'dataset_fingerprint':leakage_fingerprint,'false_positives':LEAKAGE_FALSE_POSITIVES,
                                        'confirmed':len(confirmed),'unresolved':len(unresolved)})
AUDIT_PASSED = not confirmed and not unresolved
if not AUDIT_PASSED:
    with safe_stage('record blocked audit'):
        emit_scalar('audit','confirmed_overlap_pairs',len(confirmed),0)
        emit_scalar('audit','unresolved_similarity_pairs',len(unresolved),0)
        for artifact in ['split_summary.csv','leakage_pairs.json','leakage_decisions.json','integrity.json']:
            upload_artifact('audit-'+artifact,REPORT/artifact)
        audit_zip=RUN/'audit_reports.zip'
        with zipfile.ZipFile(audit_zip,'w',compression=zipfile.ZIP_DEFLATED) as bundle:
            for asset in REPORT.rglob('*'):
                if asset.is_file():
                    if asset.suffix in {'.json','.csv','.html'}: assert_secret_free(asset.read_text())
                    bundle.write(asset,asset.relative_to(REPORT))
        upload_artifact('audit-reports',audit_zip)
        task.flush(wait_for_uploads=True)
        task.mark_failed(status_reason='Dataset audit blocked training: cross-split overlap needs resolution')
        task.close()
assert AUDIT_PASSED, f'TRAINING BLOCKED: {len(confirmed)} confirmed overlaps, {len(unresolved)} unresolved pairs. Review {review_dir}.'
print('Data gate passed.')

## 5. Memory probe and training
The probe runs two optimizer updates plus validation with the same architecture, EMA, precision, and static resolution as training. It is a feasibility check, not a guarantee for every image. Multiscale resizing is disabled to keep the memory budget predictable. OOM fallback happens only during the probe; a later training OOM saves diagnostics and requires explicit resume/reconfiguration. The final 4×2 option changes effective batch to 8 and is not expected to fix an activation-memory failure at 4×4.

Use the latest full `last.ckpt` for recovery; best `.pth` weights omit optimizer state. Epoch numbering in saved histories is one-based. The best checkpoint is selected by validation box mAP@0.50:0.95; the test loader is not called during training.

In [ ]:
from pytorch_lightning.loggers.logger import Logger as LightningLogger

class ExplicitClearMLLogger(LightningLogger):
    @property
    def name(self): return 'explicit-clearml'
    @property
    def version(self): return RUN.name
    def log_hyperparams(self, params): pass  # CFG is separately allowlisted.
    def log_metrics(self, metrics, step):
        for key,value in metrics.items():
            if torch.is_tensor(value) and value.numel()==1: value=value.detach().cpu().item()
            if isinstance(value,(int,float)): emit_scalar('training',key,value,step or 0)
    def save(self): pass
    def finalize(self,status): pass

class ExperimentCallback(Callback):
    def __init__(self,probe=False): self.probe=probe; self.started=0
    def on_train_epoch_start(self,trainer,pl_module):
        self.started=time.perf_counter(); torch.cuda.reset_peak_memory_stats()
    def on_train_batch_end(self,trainer,pl_module,outputs,batch,batch_idx):
        loss=outputs.get('loss') if isinstance(outputs,dict) else outputs
        if loss is not None and torch.is_tensor(loss) and not torch.isfinite(loss).all():
            raise FloatingPointError(f'Nonfinite training loss at epoch {trainer.current_epoch+1}, batch {batch_idx}')
    def on_validation_end(self,trainer,pl_module):
        if trainer.sanity_checking or self.probe: return
        epoch=trainer.current_epoch+1
        row={'epoch':epoch,'seconds':time.perf_counter()-self.started,
             'peak_vram_gib':torch.cuda.max_memory_allocated()/2**30}
        for key,value in trainer.callback_metrics.items():
            if torch.is_tensor(value) and value.numel()==1: value=value.detach().cpu().item()
            if isinstance(value,(int,float)):
                if not math.isfinite(float(value)): raise FloatingPointError(f'Nonfinite metric {key} at epoch {epoch}')
                row[key]=float(value)
        row['learning_rate']=trainer.optimizers[0].param_groups[0]['lr']
        with (RUN/'history.jsonl').open('a') as f: f.write(json.dumps(row)+'\n')
        for key,value in row.items():
            if key!='epoch': emit_scalar('epochs',key,value,epoch)
        print(f"Epoch {epoch}/{CFG['epochs']}: val mAP={row.get('val/mAP_50_95',float('nan')):.4f}; peak VRAM={row['peak_vram_gib']:.2f} GiB")
    def on_train_epoch_end(self,trainer,pl_module):
        if self.probe or (trainer.current_epoch+1)%5: return
        # Render a stable validation subset from current regular weights; clearly distinct from EMA best selection.
        render_progress(pl_module,trainer.current_epoch+1)
    def on_exception(self,trainer,pl_module,exception):
        msg=redact(f'epoch={trainer.current_epoch+1}, step={trainer.global_step}: {type(exception).__name__}: {exception}')
        with (RUN/'errors.log').open('a') as f: f.write(msg+'\n')
        if not self.probe:
            try: logger.report_text(msg)
            except Exception: pass

fixed_validation=sorted([r for r in records if r['split']=='valid'],key=lambda r:str(r['image_id']))[:6]

def render_progress(module,epoch):
    # Use the actual module's postprocessor, avoiding a second GPU model during training.
    from torchvision.transforms import functional as TF
    predictions={}; was_training=module.model.training
    module.model.eval()
    try:
        with torch.inference_mode():
            for row in fixed_validation:
                im=Image.open(row['path']).convert('RGB')
                tensor=TF.normalize(TF.to_tensor(im.resize((CFG['resolution'],CFG['resolution']))),
                                    [0.485,0.456,0.406],[0.229,0.224,0.225]).unsqueeze(0).to(module.device)
                with torch.autocast('cuda',dtype=torch.bfloat16): out=module.model(tensor)
                result=module.postprocess(out,torch.tensor([[row['height'],row['width']]],device=module.device))[0]
                predictions[row['image_id']]=[dict(box=b.tolist(),score=float(s)) for b,s,l in zip(result['boxes'].cpu(),result['scores'].cpu(),result['labels'].cpu()) if int(l)==0 and float(s)>=.25]
        directory=REPORT/'progress'; gallery(fixed_validation,f'epoch_{epoch:02d}_regular',predictions,.25,directory)
        if epoch: upload_artifact(f'progress-epoch-{epoch}',directory/f'epoch_{epoch:02d}_regular.png')
    finally: module.model.train(was_training)

def make_training(batch,accum,out,probe=False):
    mc=RFDETRMediumConfig(num_classes=1,resolution=CFG['resolution'],gradient_checkpointing=True,device='cuda',amp=True)
    tc=TrainConfig(dataset_dir=str(DATA),output_dir=str(out),epochs=1 if probe else CFG['epochs'],
                   batch_size=batch,grad_accum_steps=accum,eval_batch_size=CFG['eval_batch_size'],
                   lr=CFG['lr'],lr_scheduler_kwargs={'lr_drop':40},use_ema=True,eval_base_model=True,best_model_metric='map',
                   early_stopping=False,multi_scale=False,expanded_scales=False,num_workers=0,
                   checkpoint_interval=5,seed=CFG['seed'],amp_dtype='bf16',compute_val_loss=True,
                   tensorboard=False,run_test=False,progress_bar=None,class_names=['SQUIRREL'],
                   eval_max_dets=100,save_dataset_grids=False)
    module=RFDETRModelModule(mc,tc); dm=RFDETRDataModule(mc,tc)
    kwargs=dict(accelerator='gpu',devices=1,enable_model_summary=False)
    if probe: kwargs.update(limit_train_batches=accum*2,limit_val_batches=2,num_sanity_val_steps=0)
    trainer=build_trainer(tc,mc,**kwargs)
    # Keep RF-DETR's metric, EMA, and checkpoint callbacks; append our own (do not replace the list).
    trainer.callbacks.append(ExperimentCallback(probe))
    if not probe: trainer.loggers = [*trainer.loggers, ExplicitClearMLLogger()]
    return module,dm,trainer

with safe_stage('memory preflight'):
    check_space(15*2**30, RUN)
    if RESUME_RUN:
        assert (RUN/'last.ckpt').exists(), 'Full last.ckpt required.'
        selected=json.loads((RUN/'selected_batch.json').read_text())
    else:
        selected=None; attempts=[]
        for batch,accum in CFG['batch_candidates']:
            module=dm=trainer=None
            try:
                seed_everything(CFG['seed'],workers=True)
                module,dm,trainer=make_training(batch,accum,RUN/f'probe-{batch}x{accum}',True)
                torch.cuda.reset_peak_memory_stats(); trainer.fit(module,datamodule=dm)
                selected=dict(batch_size=batch,grad_accum_steps=accum,effective_batch=batch*accum,
                              peak_vram_gib=torch.cuda.max_memory_allocated()/2**30)
                attempts.append(dict(batch=batch,accum=accum,status='passed'))
            except torch.cuda.OutOfMemoryError:
                attempts.append(dict(batch=batch,accum=accum,status='CUDA OOM'))
                print(f'CUDA OOM at {batch}×{accum}; trying next configured option.')
            finally:
                del module,dm,trainer; gc.collect(); torch.cuda.empty_cache()
            save_json(RUN/'probe_attempts.json',attempts)
            if selected: break
        assert selected is not None, 'All requested batch configurations failed. Training stopped.'
        save_json(RUN/'selected_batch.json',selected)
    task.connect(selected,name='Selected batch')
    print('Selected:',selected)

In [ ]:
# Full run: this cell may take hours. Interrupting retains last.ckpt from the latest completed epoch.
assert AUDIT_PASSED
assert not (RUN/'training_complete.json').exists(), 'Training already complete; continue with evaluation.'
with safe_stage('training'):
    seed_everything(CFG['seed'],workers=True)
    module,dm,trainer=make_training(selected['batch_size'],selected['grad_accum_steps'],RUN)
    if not RESUME_RUN:
        module.to('cuda'); render_progress(module,0)
    trainer.fit(module,datamodule=dm,ckpt_path=str(RUN/'last.ckpt') if RESUME_RUN else None)
    render_progress(module,int(trainer.current_epoch))
    best_path=RUN/'checkpoint_best_total.pth'
    assert best_path.exists(), 'No best checkpoint was produced; do not evaluate/export.'
    del module,dm,trainer; gc.collect(); torch.cuda.empty_cache()
    save_json(RUN/'training_complete.json',{'best_checkpoint':best_path.name,'sha256':sha256(best_path),'max_epochs':CFG['epochs']})

In [ ]:
# Learning curves and loss components from local, explicit epoch history.
history=pd.DataFrame([json.loads(line) for line in (RUN/'history.jsonl').read_text().splitlines()])
history=history.drop_duplicates('epoch',keep='last').set_index('epoch')
csv_path=RUN/'metrics.csv'
if csv_path.exists():
    csv_metrics=pd.read_csv(csv_path)
    csv_metrics=csv_metrics.dropna(subset=['epoch'])
    csv_metrics['epoch']=csv_metrics['epoch'].astype(int)+1
    complete=csv_metrics.groupby('epoch').last().drop(columns=['step'],errors='ignore')
    history=complete.combine_first(history)
history=history.sort_index().reset_index()
for _, row in history.iterrows():
    for key,value in row.items():
        if key!='epoch' and pd.notna(value): emit_scalar('epoch_summary',key,value,int(row['epoch']))
history.to_csv(REPORT/'training_history.csv',index=False)
for name,columns in [('losses',[c for c in history if 'loss' in c]),
                     ('validation_metrics',[c for c in history if c.startswith('val/') and 'loss' not in c]),
                     ('resources',['seconds','peak_vram_gib','learning_rate'])]:
    if columns:
        fig,ax=plt.subplots(figsize=(12,5))
        history.plot(x='epoch',y=columns,ax=ax); ax.set_title(name.replace('_',' ').title()); ax.grid(alpha=.2)
        save_figure(fig,'training_'+name)
upload_artifact('training-history',REPORT/'training_history.csv')

## 6. Detection metrics, validation threshold, and held-out test
Predictions are collected at confidence ≥0.0001, at most 100 per image, without additional NMS. COCO AP uses IoUs 0.50:0.95; threshold-dependent curves/confusion use IoU 0.50 and one-to-one confidence-ordered matching. Detection has no enumerable background true-negative count; that confusion cell is N/A.

The confidence threshold maximizes validation F1 over 0.00–1.00 in steps of 0.01 (ties choose the highest threshold). Test curves describe performance but do not change this threshold. ROC/AUC is explicitly **image-level squirrel presence**, using each image's maximum squirrel score; it is undefined without both positive and negative images. This is not localization ROC.

In [ ]:
def box_iou(a,b):
    x1=max(a[0],b[0]); y1=max(a[1],b[1]); x2=min(a[2],b[2]); y2=min(a[3],b[3])
    intersection=max(0,x2-x1)*max(0,y2-y1)
    union=max(0,a[2]-a[0])*max(0,a[3]-a[1])+max(0,b[2]-b[0])*max(0,b[3]-b[1])-intersection
    return intersection/union if union>0 else 0.0

def match_detections(gt,predictions,threshold,iou=.5):
    used=set(); tp=fp=0; matched_ious=[]
    for p in sorted(predictions,key=lambda p:-p['score']):
        if p['score']<threshold: continue
        candidates=[(box_iou(p['box'],b),i) for i,b in enumerate(gt) if i not in used]
        overlap,index=max(candidates,default=(0,None))
        if index is not None and overlap>=iou:
            used.add(index); tp+=1; matched_ious.append(overlap)
        else: fp+=1
    return tp,fp,len(gt)-tp,matched_ious

def detection_metrics(rows,predictions,threshold):
    tp=fp=fn=0; ious=[]
    for row in rows:
        a,b,c,d=match_detections(row['gt'],predictions.get(row['image_id'],[]),threshold,CFG['iou'])
        tp+=a; fp+=b; fn+=c; ious.extend(d)
    precision=tp/(tp+fp) if tp+fp else 0.; recall=tp/(tp+fn) if tp+fn else 0.
    return dict(tp=tp,fp=fp,fn=fn,precision=precision,recall=recall,
                f1=2*precision*recall/(precision+recall) if precision+recall else 0.,
                mean_matched_iou=float(np.mean(ious)) if ious else None)

def presence_roc(rows,predictions):
    y=[int(bool(r['gt'])) for r in rows]
    scores=[max((p['score'] for p in predictions.get(r['image_id'],[])),default=0.) for r in rows]
    if len(set(y))<2: return None,None,None,y,scores
    fpr,tpr,_=roc_curve(y,scores)
    return fpr,tpr,float(roc_auc_score(y,scores)),y,scores

# Meaningful metric regression tests: matching, duplicate penalty, misses, and true negatives.
gt=[[0,0,10,10]]; perfect=[{'box':[0,0,10,10],'score':.9}]
assert match_detections(gt,perfect,.5)[:3]==(1,0,0)
assert match_detections(gt,perfect*2,.5)[:3]==(1,1,0)
assert match_detections(gt,[],.5)[:3]==(0,0,1)
assert match_detections([],perfect,.5)[:3]==(0,1,0)
assert match_detections([],[],.5)[:3]==(0,0,0)
assert match_detections(gt,[{'box':[20,20,30,30],'score':.9}],.5)[:3]==(0,1,1)
assert presence_roc([{'gt':[],'image_id':1}],{})[2] is None
assert presence_roc([{'gt':gt,'image_id':1},{'gt':[],'image_id':2}],{1:perfect})[2]==1.
print('Metric regression tests passed.')

In [ ]:
def predict_split(model,split):
    rows=[r for r in records if r['split']==split]; predictions={}; durations=[]
    # Warmup excluded from latency measurement.
    model.predict(Image.open(rows[0]['path']).convert('RGB'),threshold=CFG['prediction_floor'])
    for row in rows:
        im=Image.open(row['path']).convert('RGB')
        torch.cuda.synchronize(); start=time.perf_counter()
        det=model.predict(im,threshold=CFG['prediction_floor'])
        torch.cuda.synchronize(); durations.append(time.perf_counter()-start)
        pred=[dict(box=[float(x) for x in box],score=float(score)) for box,score,label in zip(det.xyxy,det.confidence,det.class_id) if int(label)==0]
        predictions[row['image_id']]=sorted(pred,key=lambda p:-p['score'])[:CFG['max_detections']]
    directory=REPORT/split; directory.mkdir(exist_ok=True)
    save_json(directory/'predictions.json',{str(k):v for k,v in predictions.items()})
    return rows,predictions,dict(mean_ms=float(np.mean(durations)*1000),p95_ms=float(np.percentile(durations,95)*1000),
                               note='Batch 1, image decode excluded; preprocessing, forward and postprocessing included.')

def coco_metrics(split,rows,predictions):
    # Normalize foreground category IDs for evaluation only; raw source annotations stay untouched.
    import copy
    doc=copy.deepcopy(splits[split]); doc['info']=doc.get('info',{})
    doc['categories']=[{'id':1,'name':'SQUIRREL'}]
    for ann in doc['annotations']:
        ann['category_id']=1; ann.setdefault('area',ann['bbox'][2]*ann['bbox'][3]); ann.setdefault('iscrowd',0)
    gt=COCO(); gt.dataset=doc; gt.createIndex()
    detections=[]
    for row in rows:
        for p in predictions[row['image_id']]:
            x1,y1,x2,y2=p['box']
            detections.append(dict(image_id=row['image_id'],category_id=1,bbox=[x1,y1,x2-x1,y2-y1],score=p['score']))
    if detections: result=gt.loadRes(detections)
    else:
        result=COCO(); result.dataset={'images':doc['images'],'categories':doc['categories'],'annotations':[]}; result.createIndex()
    evaluator=COCOeval(gt,result,'bbox'); evaluator.params.maxDets=[1,10,100]
    evaluator.evaluate(); evaluator.accumulate(); evaluator.summarize()
    names=['mAP_50_95','AP50','AP75','AP_small','AP_medium','AP_large','AR1','AR10','AR100','AR_small','AR_medium','AR_large']
    return {k:float(v) if v>=0 else None for k,v in zip(names,evaluator.stats)}

def evaluate_report(split,rows,predictions,threshold,latency):
    directory=REPORT/split; directory.mkdir(exist_ok=True)
    thresholds=np.linspace(0,1,101)
    curve=pd.DataFrame([dict(threshold=float(t),**detection_metrics(rows,predictions,float(t))) for t in thresholds])
    curve.to_csv(directory/'confidence_curves.csv',index=False)
    operating=detection_metrics(rows,predictions,threshold)
    metrics=dict(split=split,confidence_threshold=threshold,iou_threshold=CFG['iou'],operating=operating,
                 coco=coco_metrics(split,rows,predictions),latency=latency,
                 caveats=['Detection TN is undefined.','Curves truncated below confidence 0.0001, at 100 detections/image.',
                          'Source/burst independence cannot be guaranteed without provenance.'])
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    for ax,key in zip(axes,['precision','recall','f1']):
        ax.plot(curve.threshold,curve[key]); ax.axvline(threshold,color='red',ls='--'); ax.set(xlabel='Confidence',ylabel=key,title=f'{key} vs confidence',ylim=(0,1.02))
    save_figure(fig,'precision_recall_f1_confidence',directory)
    fig,ax=plt.subplots(); ax.plot(curve.recall,curve.precision); ax.set(xlabel='Recall',ylabel='Precision',title='Detection PR · IoU 0.50',xlim=(0,1),ylim=(0,1.02))
    save_figure(fig,'detection_pr',directory)
    matrix=np.array([[operating['tp'],operating['fn']],[operating['fp'],np.nan]])
    fig,ax=plt.subplots(); ax.imshow(np.ma.masked_invalid(matrix),cmap='Blues')
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Squirrel','Background'],yticklabels=['Squirrel','Background'],xlabel='Predicted',ylabel='Actual',title='Detection confusion · TN undefined')
    for (i,j),v in np.ndenumerate(matrix): ax.text(j,i,'N/A' if np.isnan(v) else str(int(v)),ha='center',va='center')
    save_figure(fig,'detection_confusion',directory)
    fpr,tpr,auc,y,scores=presence_roc(rows,predictions); metrics['image_presence_roc_auc']=auc
    fig,ax=plt.subplots()
    if auc is None:
        ax.text(.5,.5,'ROC/AUC undefined: both image classes required',ha='center',wrap=True); ax.axis('off')
    else:
        ax.plot(fpr,tpr,label=f'AUC = {auc:.4f}'); ax.plot([0,1],[0,1],'--'); ax.legend()
        ax.set(xlabel='False-positive rate',ylabel='True-positive rate',title='Image-level squirrel-presence ROC')
    save_figure(fig,'image_presence_roc',directory)
    cm=confusion_matrix(y,[int(s>=threshold) for s in scores],labels=[0,1]); metrics['image_presence_confusion']=cm.tolist()
    fig,ax=plt.subplots(); ax.imshow(cm,cmap='Blues')
    ax.set(xticks=[0,1],yticks=[0,1],xticklabels=['Absent','Present'],yticklabels=['Absent','Present'],xlabel='Predicted',ylabel='Actual',title='Image-presence confusion')
    for (i,j),v in np.ndenumerate(cm): ax.text(j,i,str(v),ha='center',va='center')
    save_figure(fig,'image_presence_confusion',directory)
    errors={'false_positives':[],'missed_objects':[],'poor_localization':[],'successes':[]}
    for row in rows:
        tp,fp,fn,_=match_detections(row['gt'],predictions[row['image_id']],threshold)
        if fp: errors['false_positives'].append(row)
        if fn: errors['missed_objects'].append(row)
        if any(.1<=box_iou(p['box'],b)<.5 for p in predictions[row['image_id']] if p['score']>=threshold for b in row['gt']): errors['poor_localization'].append(row)
        if tp and not fp and not fn: errors['successes'].append(row)
    for name,items in errors.items(): gallery(items[:9],name,predictions,threshold,directory)
    save_json(directory/'metrics.json',metrics); html_report(directory,f'{split.title()} evaluation',metrics)
    for name,value in metrics['coco'].items():
        if value is not None: emit_scalar(split,name,value,0)
    for name,value in operating.items():
        if value is not None: emit_scalar(split,name,value,0)
    if auc is not None: emit_scalar(split,'image_presence_roc_auc',auc,0)
    for path in directory.iterdir(): upload_artifact(f'{split}-{path.name}',path)
    return metrics

with safe_stage('validation'):
    best_path=RUN/'checkpoint_best_total.pth'
    assert (RUN/'training_complete.json').exists(), 'Complete training before evaluation.'
    model=RFDETRMedium(pretrain_weights=str(best_path),device='cuda')
    val_rows,val_predictions,val_latency=predict_split(model,'valid')
    candidates=[(detection_metrics(val_rows,val_predictions,float(t))['f1'],float(t)) for t in np.linspace(0,1,101)]
    _,threshold=max(candidates)
    save_json(RUN/'decision_threshold.json',{'threshold':threshold,'selection':'maximum validation F1 at IoU 0.50; ties choose highest','checkpoint_sha256':sha256(best_path)})
    validation_metrics=evaluate_report('valid',val_rows,val_predictions,threshold,val_latency)
    print('Frozen validation threshold:',threshold)

In [ ]:
# Test is evaluated only after checkpoint and confidence threshold have been frozen.
with safe_stage('test evaluation'):
    decision=json.loads((RUN/'decision_threshold.json').read_text())
    assert decision['checkpoint_sha256']==sha256(best_path)
    threshold=decision['threshold']
    test_rows,test_predictions,test_latency=predict_split(model,'test')
    test_metrics=evaluate_report('test',test_rows,test_predictions,threshold,test_latency)
    comparison=pd.DataFrame({'validation':validation_metrics['coco'],'test':test_metrics['coco']})
    comparison.to_csv(REPORT/'validation_test_comparison.csv'); display(comparison)
    html_report(REPORT,'Squirrel detection — final report',{'dataset':eda_summary,'validation':validation_metrics,'test':test_metrics,
                'subreports':['valid/report.html','test/report.html','leakage_review/report.html'],
                'outcome':'Measured baseline; no predefined performance target. Inspect localization errors and split limitations.'})

## 7. Export, verify, and close
The chosen best weights are copied to the stable output folder, while original checkpoints are retained for reproducibility. The export includes a model card, class mapping, threshold, source provenance, selected epoch, metrics, and checksum. Existing stable exports are archived before replacement.

In [ ]:
with safe_stage('export'):
    destination=ROOT/'output/model_weights'; destination.mkdir(parents=True,exist_ok=True)
    target=destination/'best_squirrel_rfdetr_medium.pth'
    if target.exists():
        archive_dir=destination/'previous'/datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
        archive_dir.mkdir(parents=True,exist_ok=True)
        for p in destination.iterdir():
            if p.is_file(): shutil.copy2(p,archive_dir/p.name)
    temporary=destination/'best_squirrel_rfdetr_medium.pth.partial'
    shutil.copy2(best_path,temporary); assert sha256(temporary)==sha256(best_path); temporary.replace(target)
    sample=Image.open(val_rows[0]['path']).convert('RGB')
    original=model.predict(sample,threshold=threshold)
    del model; gc.collect(); torch.cuda.empty_cache()
    exported=RFDETRMedium(pretrain_weights=str(target),device='cuda')
    reloaded=exported.predict(sample,threshold=threshold)
    np.testing.assert_allclose(original.xyxy,reloaded.xyxy,rtol=1e-4,atol=1e-3)
    np.testing.assert_allclose(original.confidence,reloaded.confidence,rtol=1e-4,atol=1e-5)
    np.testing.assert_array_equal(original.class_id,reloaded.class_id)
    checkpoint=torch.load(best_path,map_location='cpu',weights_only=False)  # trusted locally generated checkpoint only
    metadata=dict(model='RFDETRMedium',class_mapping={'0':'SQUIRREL'},resolution=CFG['resolution'],
                  confidence_threshold=threshold,epoch_zero_based=checkpoint.get('epoch'),dataset=manifest,
                  environment=environment,selected_batch=selected,validation=validation_metrics,test=test_metrics,
                  sha256=sha256(target),clearml_task_id=task.id,run_directory=str(RUN))
    del checkpoint
    save_json(destination/'model_metadata.json',metadata)
    model_card='# Squirrel detection model\nRF-DETR Medium, one SQUIRREL class. Trained on Root and Nut / Roboflow Universe v6 (CC BY 4.0).\nSee model_metadata.json for actual validation/test metrics, confidence threshold, checkpoint epoch,\ndependency versions, dataset hashes and run provenance. Not a species or individual-ID classifier.\nTemporal/burst independence is not guaranteed without source metadata. Evaluate on new camera/site\nimages before claiming generalization. Green boxes in reports are truth; red boxes are predictions.\n'
    (destination/'MODEL_CARD.md').write_text(model_card)
    upload_artifact('model-metadata',destination/'model_metadata.json')
    output_model=OutputModel(task=task,framework='PyTorch',name='Squirrel RF-DETR Medium',label_enumeration={'SQUIRREL':0})
    output_model.update_weights(weights_filename=str(target),auto_delete_file=False)
    for path in REPORT.glob('*'):
        if path.is_file(): upload_artifact('report-'+path.name,path)
    for name in ['console.log','errors.log','environment.json','probe_attempts.json','selected_batch.json']:
        if (RUN/name).exists(): upload_artifact(name,RUN/name)
    pending=RUN/'pending_scalars.jsonl'
    if pending.exists():
        events=[json.loads(line) for line in pending.read_text().splitlines()]
        for event in events: logger.report_scalar(**event)
        pending.unlink()
    pending=RUN/'pending_artifacts.jsonl'
    if pending.exists():
        events=[json.loads(line) for line in pending.read_text().splitlines()]
        for event in events: task.upload_artifact(name=event['name'],artifact_object=event['path'],wait_on_upload=True)
        pending.unlink()
    report_zip=RUN/'reports.zip'
    with zipfile.ZipFile(report_zip,'w',compression=zipfile.ZIP_DEFLATED) as bundle:
        for asset in REPORT.rglob('*'):
            if asset.is_file():
                if asset.suffix in {'.json','.jsonl','.csv','.html','.log','.txt'}: assert_secret_free(asset.read_text())
                bundle.write(asset,asset.relative_to(REPORT))
    upload_artifact('complete-reports',report_zip)
    task.flush(wait_for_uploads=True)
    save_json(RUN/'workflow_complete.json',{'weights':str(target),'sha256':sha256(target),'clearml_task_id':task.id})
    task.close()
    print('Verified best weights:',target)

In [ ]:
# Inference example; change IMAGE_PATH to your own local image.
IMAGE_PATH = Path(val_rows[0]['path'])
with safe_stage('exported-model inference'):
    detections=exported.predict(Image.open(IMAGE_PATH).convert('RGB'),threshold=threshold)
    preview={'path':str(IMAGE_PATH),'gt':[]}
    predictions=[dict(box=b.tolist(),score=float(s)) for b,s,l in zip(detections.xyxy,detections.confidence,detections.class_id) if int(l)==0]
    display(draw_sample(preview,predictions,threshold))

## References and interpretation
- [Dataset v6 and attribution](https://universe.roboflow.com/root-and-nut/squirrel-re-id-training-v1-fzpbr/dataset/6)
- [RF-DETR training parameters](https://rfdetr.roboflow.com/latest/learn/train/training-parameters/)
- [RF-DETR custom training API](https://rfdetr.roboflow.com/latest/learn/train/customization/)
- [ClearML task controls](https://clear.ml/docs/latest/docs/references/sdk/task/)

Use the locked package versions: online `latest` documentation may change. Report real achieved scores, failures, negative-image availability, and possible source dependence. High validation scores alone do not establish performance on new environments. ROC/AUC here answers image presence; COCO AP and detection PR assess localization.